## 2.

In [7]:
import pandas as pd

df_sites = pd.read_csv(r"/Users/damieneschimese/Desktop/Formation Data Analyst/tp-reeval-061126-dossier-apprenants/data/sites.csv")
df_materiel = pd.read_csv(r"/Users/damieneschimese/Desktop/Formation Data Analyst/tp-reeval-061126-dossier-apprenants/data/materiel.csv")
df_associations = pd.read_csv(r"/Users/damieneschimese/Desktop/Formation Data Analyst/tp-reeval-061126-dossier-apprenants/data/associations.csv") # ou usagers.csv selon ton fichier
df_reservations = pd.read_csv(r"/Users/damieneschimese/Desktop/Formation Data Analyst/tp-reeval-061126-dossier-apprenants/data/reservations.csv")

df_reservations["date_debut"] = pd.to_datetime(df_reservations["date_debut"], format="%Y-%m-%d")
df_reservations["date_fin_prevue"] = pd.to_datetime(df_reservations["date_fin_prevue"], format="%Y-%m-%d")
df_reservations["date_retour_reelle"] = pd.to_datetime(df_reservations["date_retour_reelle"], format="%Y-%m-%d")



## 3.

In [13]:
print("Matériel")
print(df_materiel.shape)
print(df_materiel.info)
print(df_materiel.isna().sum())

print("")
print("Réservations")
print(df_reservations.shape)
print(df_reservations.info)
print(df_reservations.isna().sum())

Matériel
(367, 8)
<bound method DataFrame.info of      id_materiel reference            designation        categorie       etat  \
0            198  MAT-0198      Chapiteau 10x15 m  Tente et barnum      moyen   
1            174  MAT-0174      Chapiteau 10x15 m  Tente et barnum        bon   
2            308  MAT-0308         Rubalise 500 m         Securite        bon   
3            213  MAT-0213              Rampe LED        Eclairage        bon   
4            231  MAT-0231        Mat d'eclairage        Eclairage        bon   
..           ...       ...                    ...              ...        ...   
362          348  MAT-0348            Plancha gaz          Cuisine  a reviser   
363          118  MAT-0118                Pupitre         Mobilier        bon   
364          311  MAT-0311  Cone de signalisation         Securite       neuf   
365          284  MAT-0284        Extincteur 6 kg         Securite        bon   
366          133  MAT-0133          Estrade 2x1 m         M

## 4.

In [14]:
print(df_materiel["valeur_achat"].dtype)

str


Il y a des entrées vides et quelques nombres avec des séparateurs de décimaux ',' au lieu du '.'

In [15]:
df_materiel["valeur_achat"] = pd.to_numeric(df_materiel["valeur_achat"].astype(str).str.replace(',', '.'), errors='coerce')
print(df_materiel["valeur_achat"].dtype)

float64


## 5.

In [23]:
print("Top 10 matériel le plus cher")
top_10_chers = df_materiel.sort_values(by="valeur_achat", ascending=False).head(10)
print(top_10_chers[["reference", "designation", "categorie", "valeur_achat"]])

print("")
print("Nb par catégorie")
nb_materiel_cat = df_materiel["categorie"].value_counts()
print(nb_materiel_cat)

print("")
print("Réservations par site")
stats_sites = df_reservations.groupby("code_site").agg(
    nombre_reservations=('code_site', 'count'),
    quantite_totale=('quantite', 'sum')
).sort_values(by="nombre_reservations", ascending=False)
print(stats_sites)

Top 10 matériel le plus cher
    reference               designation        categorie  valeur_achat
332  MAT-0202            Plancher 3x3 m  Tente et barnum       1484.04
235  MAT-0176  Tente de reception 5x8 m  Tente et barnum       1473.11
323  MAT-0194            Plancher 3x3 m  Tente et barnum       1447.45
333  MAT-0210         Chapiteau 10x15 m  Tente et barnum       1444.67
356  MAT-0189            Plancher 3x3 m  Tente et barnum       1401.25
153  MAT-0169         Chapiteau 10x15 m  Tente et barnum       1318.60
277  MAT-0168              Barnum 3x3 m  Tente et barnum       1311.53
20   MAT-0170              Barnum 3x6 m  Tente et barnum       1292.07
22   MAT-0198         Chapiteau 10x15 m  Tente et barnum       1276.35
0    MAT-0198         Chapiteau 10x15 m  Tente et barnum       1276.35

Nb par catégorie
categorie
Mobilier           111
Securite            69
Sonorisation        53
Tente et barnum     46
Eclairage           44
Cuisine             33
SONORISATION         3
 

## 6.

In [25]:
df_complet = df_reservations.join(
    df_materiel.set_index("id_materiel"), 
    on="id_materiel", 
    how="left",
    lsuffix="_res",
    rsuffix="_mat"
).join(
    df_associations.set_index("id_association"), 
    on="id_association", 
    how="left",
    rsuffix="_assoc"
)
print(f"Nombre total de lignes dans df_complet : {len(df_complet)}")

nb_sans_materiel = df_complet["designation"].isna().sum()
print(f"Nombre de lignes sans matériel correspondant : {nb_sans_materiel}")

Nombre total de lignes dans df_complet : 4725
Nombre de lignes sans matériel correspondant : 21


## 7.

In [26]:
import os

os.makedirs("sorties", exist_ok=True)

top10_materiel = (
    df_complet.groupby(["id_materiel", "designation"])
    .size()
    .reset_index(name="nombre_reservations")
    .sort_values(by="nombre_reservations", ascending=False)
    .head(10)
)

top10_materiel.to_csv("sorties/top10_materiel.csv", index=False, encoding="utf-8")

print("--- Top 10 des matériels les plus réservés ---")
print(top10_materiel)

--- Top 10 des matériels les plus réservés ---
     id_materiel                   designation  nombre_reservations
3              4             Caisson de basses                   70
42            43       Table de mixage 8 voies                   48
203          204                  Barnum 3x3 m                   47
289          290          Panneau de deviation                   45
253          254     Guirlande guinguette 20 m                   45
332          333   Four a raclette 8 personnes                   43
129          130                Chaise pliante                   43
115          116          Table pliante 180 cm                   42
262          263                Rubalise 500 m                   42
20            21  Enceinte amplifiee 12 pouces                   41
